# Day 6 — PySpark: Window Functions Part 1

> **Topics:** `Window.partitionBy` · `row_number` · `rank` · `dense_rank` · Top-N per group  
> **Run order:** top to bottom

In [ ]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day6_Window_Part1') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# Sales data — same as SQL notebook (Bob & Carol tie in North/month1)
sales_data = [
    (1, 'Alice', 'North', 1, 5000.0),
    (2, 'Bob',   'North', 1, 4200.0),
    (3, 'Carol', 'North', 1, 4200.0),   # tie with Bob
    (4, 'Dave',  'North', 1, 3800.0),
    (5, 'Eve',   'South', 1, 6100.0),
    (6, 'Frank', 'South', 1, 5500.0),
    (7, 'Grace', 'South', 1, 5500.0),   # tie with Frank
    (8, 'Hank',  'South', 1, 4000.0),
    (1, 'Alice', 'North', 2, 5300.0),
    (2, 'Bob',   'North', 2, 4800.0),
    (3, 'Carol', 'North', 2, 4000.0),
    (5, 'Eve',   'South', 2, 5900.0),
    (6, 'Frank', 'South', 2, 6300.0),   # Frank tops South in month 2
    (7, 'Grace', 'South', 2, 5100.0),
]
schema = StructType([
    StructField('emp_id',   IntegerType(), False),
    StructField('emp_name', StringType(),  True),
    StructField('region',   StringType(),  True),
    StructField('month',    IntegerType(), True),
    StructField('revenue',  DoubleType(),  True),
])
df_sales = spark.createDataFrame(sales_data, schema=schema)
print('Rows:', df_sales.count())
df_sales.orderBy('region', 'month', F.desc('revenue')).show()

---
## 1. Window Spec — The Foundation

`Window.partitionBy('region', 'month').orderBy(F.desc('revenue'))`  
= OVER (PARTITION BY region, month ORDER BY revenue DESC)

In [ ]:
# Define the window once — reuse across multiple withColumn() calls
w = Window.partitionBy('region', 'month').orderBy(F.desc('revenue'))

# Apply all three functions at once
df_ranked = (
    df_sales
    .withColumn('rn',   F.row_number().over(w))
    .withColumn('rnk',  F.rank().over(w))
    .withColumn('drnk', F.dense_rank().over(w))
    .orderBy('region', 'month', 'rnk')
)
df_ranked.show()

---
## 2. See the Difference — North Month 1 (Tie at 4200)

In [ ]:
# Filter to North/month1 to see tie behavior clearly
df_north1 = df_ranked.filter(
    (F.col('region') == 'North') & (F.col('month') == 1)
).orderBy('rnk')

print('North / Month 1 — tie at revenue=4200:')
df_north1.select('emp_name', 'revenue', 'rn', 'rnk', 'drnk').show()

print('Observations:')
print('  row_number: Bob=2 Carol=3  (always unique, arbitrary tiebreak)')
print('  rank:       Bob=2 Carol=2, Dave=4  (gap — 3 is skipped)')
print('  dense_rank: Bob=2 Carol=2, Dave=3  (no gap)')

---
## 3. Top-N per Group — Filter After withColumn()

In [ ]:
# Top 2 per region in month 1 — use dense_rank so ties are included
w_month1 = Window.partitionBy('region').orderBy(F.desc('revenue'))

df_top2 = (
    df_sales
    .filter(F.col('month') == 1)
    .withColumn('drnk', F.dense_rank().over(w_month1))
    .filter(F.col('drnk') <= 2)
    .orderBy('region', 'drnk')
)

print('Top 2 per region (month 1) — ties BOTH included:')
df_top2.show()
# North: Alice(1), Bob(2), Carol(2) — 3 rows due to tie
# South: Eve(1), Frank(2), Grace(2) — 3 rows due to tie

---
## 4. ROW_NUMBER for Deduplication Pattern

Keep only the highest-revenue record per employee (across months).

In [ ]:
# Keep best month per employee
w_emp = Window.partitionBy('emp_id').orderBy(F.desc('revenue'))

df_best = (
    df_sales
    .withColumn('rn', F.row_number().over(w_emp))
    .filter(F.col('rn') == 1)          # keep only best row per emp_id
    .drop('rn')
    .orderBy('emp_id')
)

print('Best month per employee (ROW_NUMBER dedup):')
df_best.show()

---
## 5. Window Without partitionBy — Global Ranking

In [ ]:
# No partitionBy → rank globally across all rows
w_global = Window.orderBy(F.desc('revenue'))

df_global = (
    df_sales
    .withColumn('global_rank', F.dense_rank().over(w_global))
    .orderBy('global_rank')
    .limit(5)
)
print('Global top 5 by revenue:')
df_global.show()

---
## Quick Reference

```python
from pyspark.sql import Window, functions as F

# Window spec
w = Window.partitionBy('region').orderBy(F.desc('revenue'))

# Apply functions
.withColumn('rn',   F.row_number().over(w))   # unique
.withColumn('rnk',  F.rank().over(w))          # gaps on tie
.withColumn('drnk', F.dense_rank().over(w))    # no gaps

# Top-N filter
.filter(F.col('drnk') <= 2)

# Dedup pattern
.withColumn('rn', F.row_number().over(w)).filter(F.col('rn') == 1)
```

| Function | Ties | Gap | Use for |
|----------|------|-----|---------|
| `row_number()` | No | — | Dedup, pick one per group |
| `rank()` | Yes | Yes | Olympic-style ranking |
| `dense_rank()` | Yes | No | Top-N filtering |

In [ ]:
spark.stop()
print('Spark stopped.')